## 1. Outcomes and model scores

For a binary labeled outcome, $1$ is the positive class and $0$ is the negative class. A classifier gives each row a score. A larger score means stronger evidence that the row belongs to outcome $1$.

Consider this four-row holdout example:

In [3]:
#| label: simple-auroc-holdout

import pandas as pd
from IPython.display import display

holdout = pd.DataFrame(
    {
        "row": ["A", "B", "C", "D"],
        "outcome": [1, 1, 0, 0],
        "model score": [0.90, 0.40, 0.70, 0.20],
    }
)
display(holdout)

,row,outcome,model score
0,A,1,0.9
1,B,1,0.4
2,C,0,0.7
3,D,0,0.2


## 2. Compare every positive–negative pair

Let $u^+$ be a randomly selected holdout row with outcome $1$, let $u^-$ be a randomly selected holdout row with outcome $0$, and let $s(u)$ be the model score for row $u$. AUROC is

$$
\mathrm{AUROC}
=P\!\left(s(u^+)>s(u^-)\right)
+\frac12P\!\left(s(u^+)=s(u^-)\right).
$$

The first probability is the chance that the outcome-$1$ row receives the higher score, which is a correct ranking. The second probability is the chance of a tied score, and the factor $1/2$ gives each tie half credit. If the outcome-$1$ row receives the lower score, that pair contributes zero.

Therefore, the equation is the expected ranking credit for one randomly selected positive–negative pair. In a finite holdout, calculate it directly by pairing every outcome-$1$ row with every outcome-$0$ row and averaging their credits.

In [4]:
#| label: pairwise-auroc-ranking

positive_rows = holdout.loc[holdout["outcome"] == 1]
negative_rows = holdout.loc[holdout["outcome"] == 0]
pair_rows = []

for _, positive in positive_rows.iterrows():
    for _, negative in negative_rows.iterrows():
        positive_score = positive["model score"]
        negative_score = negative["model score"]
        if positive_score > negative_score:
            credit = 1.0
        elif positive_score == negative_score:
            credit = 0.5
        else:
            credit = 0.0
        pair_rows.append(
            {
                "outcome-1 row": positive["row"],
                "outcome-0 row": negative["row"],
                "comparison": f"{positive_score:.2f} vs {negative_score:.2f}",
                "credit": credit,
            }
        )

pairs = pd.DataFrame(pair_rows)
AUROC = pairs["credit"].mean()
display(pairs)
print(f"AUROC = {AUROC:.2f}")
assert AUROC == 0.75

,outcome-1 row,outcome-0 row,comparison,credit
0,A,C,0.90 vs 0.70,1.0
1,A,D,0.90 vs 0.20,1.0
2,B,C,0.40 vs 0.70,0.0
3,B,D,0.40 vs 0.20,1.0


AUROC = 0.75


Row A ranks above C and D. Row B ranks above D but below C. The model therefore ranks three of the four positive–negative pairs correctly:

$$
\mathrm{AUROC}=\frac{3}{4}=0.75.
$$

## 3. Interpret AUROC

- An AUROC of $1$ means every outcome-$1$ row receives a higher score than every outcome-$0$ row.
- An AUROC of $0.5$ means the scores provide chance-level ranking.
- An AUROC below $0.5$ means the ranking tends to be reversed.

AUROC does not require a cutoff that converts scores into predicted labels. It evaluates the ordering produced by the scores themselves.

## Takeaway

AUROC is the average credit obtained by comparing every outcome-$1$ row with every outcome-$0$ row: one point when the positive row ranks higher, one-half for a tie, and zero when it ranks lower.

Return to [7. Utility Metric: Independent-Holdout AUROC Difference](7.0-utility-metric-independent-holdout-auroc-difference.ipynb).